# 📈 Stock Data Analysis

這個 Notebook 專門用來調用 `database` 模組中的資料，並將其轉換為 Pandas DataFrame 以供分析。

### 前置準備
請確保您已安裝以下套件：
```bash
pip install pandas plotly jupyterlab
```

In [ ]:
pwd

In [ ]:
import pandas as pd
import sys
import os

# 1. 設定路徑：確保可以 import 專案根目錄的模組
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

try:
    from database.client import get_db_client
    print("✅ 成功載入 database 模組")
except ImportError as e:
    print(f"❌ 載入失敗: {e}")
    print("   請確認 database/client.py 存在且路徑正確。")

✅ 成功載入 database 模組


In [ ]:
def get_table_as_df(table_name):
    """
    使用專案的 db client 讀取整張資料表
    """
    print(f"📥 正在讀取資料表: {table_name} ...")
    try:
        with get_db_client() as db:
            # 根據 crawlers/price.py 的用法，db.fetch 回傳 DataFrame
            df = db.fetch(table_name)
            print(f"✅ 讀取完成！共 {len(df)} 筆資料")
            return df
    except Exception as e:
        print(f"❌ 讀取錯誤: {e}")
        return pd.DataFrame()

## 1. 讀取台股資料 (TWStock)

In [ ]:
df_tw = get_table_as_df("tw_daily_prices")

# 預覽前 5 筆
df_tw.head()

📥 正在讀取資料表: tw_daily_prices ...
✅ 讀取完成！共 523468 筆資料


,date,stock_id,stock_name,market,open,high,low,close,volume,turnover,transactions
0,2026-01-30,0050,元大台灣50,TWSE,73.15,73.20,72.30,72.60,175275799,1.273881e+10,241546
1,2026-01-30,0051,元大中型100,TWSE,106.55,106.75,105.25,106.70,89671,9.513773e+06,959
2,2026-01-30,0052,富邦科技,TWSE,43.50,43.54,43.10,43.28,76077198,3.296105e+09,56146
3,2026-01-30,0053,元大電子,TWSE,160.75,161.55,159.80,161.55,23978,3.852667e+06,262
4,2026-01-30,0055,元大MSCI金融,TWSE,32.44,32.44,31.98,32.03,188795,6.074041e+06,342


## 2. 讀取美股資料 (USStock)

In [42]:
df_us = get_table_as_df("us_daily_prices")

# 預覽前 5 筆
df_us.head()

📥 正在讀取資料表: us_daily_prices ...
✅ 讀取完成！共 3795 筆資料


,date,ticker,open,high,low,close,adj_close,volume
0,2025-03-13,^SOX,4485.830078,4548.540039,4413.680176,4453.240234,4453.240234,0
1,2025-04-07,^SOX,3460.449951,3896.070068,3388.620117,3694.949951,3694.949951,0
2,2025-04-08,^SOX,3863.699951,3902.530029,3484.270020,3562.939941,3562.939941,0
3,2025-04-09,^SOX,3609.169922,4265.020020,3572.479980,4230.450195,4230.450195,0
4,2025-04-10,^SOX,4020.129883,4036.870117,3757.520020,3893.300049,3893.300049,0


## 3. 簡單視覺化範例 (Plotly)
這裡示範如何畫出 K 線圖。

In [43]:
import plotly.graph_objects as go

def plot_stock(df, stock_id_or_ticker):
    if df.empty:
        print("⚠️ 資料集為空，無法繪圖")
        return
    
    # 判斷欄位名稱 (台股用 stock_id, 美股用 ticker)
    col_name = 'stock_id' if 'stock_id' in df.columns else 'ticker'
    
    # 篩選資料
    target_df = df[df[col_name] == stock_id_or_ticker].copy()
    
    if target_df.empty:
        print(f"⚠️ 找不到代號 {stock_id_or_ticker} 的資料")
        return
        
    # 確保日期格式
    target_df['date'] = pd.to_datetime(target_df['date'])
    target_df = target_df.sort_values('date')
    
    # 繪圖
    fig = go.Figure(data=[go.Candlestick(
        x=target_df['date'],
        open=target_df['open'],
        high=target_df['high'],
        low=target_df['low'],
        close=target_df['close'],
        name=stock_id_or_ticker
    )])
    
    fig.update_layout(
        title=f"{stock_id_or_ticker} 股價走勢",
        yaxis_title="Price",
        xaxis_title="Date",
        template="plotly_dark"
    )
    fig.show()

# 測試：畫出台積電 (2330) 或其他存在的股票
# 請確保資料庫內有該股票資料
plot_stock(df_tw, "2330")

## 4. 計算技術指標 (Technical Indicators)
這裡我們將引用 `indicators.py` 中的邏輯，自動計算均線 (MA)、乖離率等指標。

In [44]:
try:
    from indicators import calculate_technical_indicators
    print("✅ 成功載入 indicators 模組")
except ImportError:
    print("❌ 找不到 indicators.py，請確認檔案位於專案根目錄")

# 確保 df_tw 已經讀取 (如果前面沒跑，這裡重讀一次)
if 'df_tw' not in locals() or df_tw.empty:
    df_tw = get_table_as_df("tw_daily_prices")

if not df_tw.empty:
    # 🔥 修改點：不再只篩選一檔，而是對「全市場」股票進行計算
    print(f"📊 正在為全市場 {df_tw['stock_id'].nunique()} 檔股票計算技術指標 (這可能需要一點時間)...")
    
    # indicators.py 裡的邏輯已經支援 groupby('stock_id')，所以可以直接丟整個 df 進去
    df_all_indicators = calculate_technical_indicators(df_tw)
    print("✅ 計算完成！")
    
    # --- 新增：保存所有計算結果 (CSV) ---
    import os
    save_dir = "output_data"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    # 取得最新日期作為檔名
    last_dt = df_all_indicators['date'].max()
    date_str = last_dt.strftime('%Y-%m-%d') if hasattr(last_dt, 'strftime') else str(last_dt)
    csv_path = f"{save_dir}/all_technical_indicators_{date_str}.csv"
    
    print(f"💾 正在將全市場指標存檔至: {csv_path}")
    # encoding='utf-8-sig' 是為了讓 Excel 開啟時中文不亂碼
    df_all_indicators.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print("✅ 存檔完成！")
    
    # --- 範例應用：找出強勢股 ---
    # 1. 取出最近一天的資料
    latest_date = df_all_indicators['date'].max()
    df_today = df_all_indicators[df_all_indicators['date'] == latest_date].copy()
    
    print(f"\n🏆 {latest_date} 市場掃描結果:")
    
    # 2. 策略 A: 量能爆發 (含防妖濾網)
    # 計算 20日均成交值 (億) - 近似值: 20日均量 * 目前股價
    df_today['avg_turnover_20d'] = (df_today['avg_vol_baseline'] * df_today['close']) / 100000000
    
    # 防妖濾網: 均量>2000張, 均值>1億, 股價>20元, 多頭排列
    mask = (
        (df_today['close'] > df_today['price_ma20']) & 
        (df_today['vol_ratio'] > 2.0) & 
        (df_today['avg_vol_baseline'] > 2000000) & 
        (df_today['avg_turnover_20d'] > 1.0) & 
        (df_today['close'] > 20)
    )
    
    top_volume = df_today[mask].sort_values('vol_ratio', ascending=False).head(5)
    print("\n🛡️ [防妖濾網] 量能爆發且流動性佳 Top 5:")
    display(top_volume[['stock_id', 'stock_name', 'close', 'vol_ratio', 'avg_turnover_20d', 'bias']])
    
    # 3. 策略 B: 乖離率過大 (正乖離 > 10%，可能過熱)
    overheated = df_today[df_today['bias'] > 0.1].sort_values('bias', ascending=False).head(5)
    print("\n⚠️ 短線過熱 (乖離率 > 10%) Top 5:")
    display(overheated[['stock_id', 'stock_name', 'close', 'bias', 'volatility']])
    
else:
    print("⚠️ 找不到資料，請先確認資料庫是否有該股票數據")

✅ 成功載入 indicators 模組
📊 正在為全市場 2107 檔股票計算技術指標 (這可能需要一點時間)...


c:\Users\Qoo\Desktop\my_workspace\reason-stock-agent\indicators.py:32: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



✅ 計算完成！
💾 正在將全市場指標存檔至: output_data/all_technical_indicators_2026-02-04.csv
✅ 存檔完成！

🏆 2026-02-04 市場掃描結果:

🛡️ [防妖濾網] 量能爆發且流動性佳 Top 5:


,stock_id,stock_name,close,vol_ratio,avg_turnover_20d,bias
522954,4991,環宇-KY,265.50,7.308899,6.383254,0.129907
521782,2355,敬鵬,43.75,5.444466,4.762890,0.201923
521821,2413,環科,48.40,4.479488,2.273223,0.064438
521646,1711,永光,23.90,4.230637,3.599413,0.075608
521859,2464,盟立,72.00,3.638631,3.569172,0.062652



⚠️ 短線過熱 (乖離率 > 10%) Top 5:


,stock_id,stock_name,close,bias,volatility
522329,6285,啟碁,207.5,0.454609,0.058911
521791,2367,燿華,59.5,0.399259,0.068321
522449,6949,沛爾生醫-創,620.0,0.374647,0.021967
522393,6715,嘉基,150.0,0.324796,0.078408
522343,6451,訊芯-KY,208.0,0.316664,0.089998


In [ ]:
# (進階) 畫出包含均線的 K 線圖
import plotly.graph_objects as go

# 設定你想畫的股票代號
target_stock_id = "2330"  # 可以改成上面掃描出來的強勢股代號

if 'df_all_indicators' in locals() and not df_all_indicators.empty:
    # 從計算好的大表中篩選出這一檔
    df_plot = df_all_indicators[df_all_indicators['stock_id'] == target_stock_id].copy()
    
    if not df_plot.empty:
        fig = go.Figure()

        # K線
        fig.add_trace(go.Candlestick(
            x=df_plot['date'],
            open=df_plot['open'], high=df_plot['high'],
            low=df_plot['low'], close=df_plot['close'],
            name='K線'
        ))

        # MA5 (黃線)
        fig.add_trace(go.Scatter(
            x=df_plot['date'], y=df_plot['price_ma5'],
            mode='lines', name='MA5', line=dict(color='orange', width=1)
        ))

        # MA20 (紫線)
        fig.add_trace(go.Scatter(
            x=df_plot['date'], y=df_plot['price_ma20'],
            mode='lines', name='MA20', line=dict(color='purple', width=1)
        ))

        fig.update_layout(
            title=f"{target_stock_id} 技術分析圖 (含均線)",
            yaxis_title="Price",
            template="plotly_dark",
            height=600
        )
        fig.show()
    else:
        print(f"⚠️ 找不到 {target_stock_id} 的資料")
else:
    print("⚠️ 請先執行上一個 Cell 計算技術指標")